# FanSphere-AI · Stage 3 — Audience Sentiment & Engagement

**Goal:** Combine StatsBomb football fixtures (Stage 2) with historical Reddit fan discussions to produce a unified engagement signal per match.

**Pipeline:**

```
stage2_matches.csv
        │
        ▼
reddit_comments.parquet  ──►  link_comments_to_matches  ──►  sentiment  ──►  aggregate  ──►  join with stage 2
(Pushshift archives)
```

**Architectural note.** The ingestion source (Pushshift archives, here) is hidden behind an abstract `BaseRedditConnector`. Linking / sentiment / engagement modules consume a canonical `Comment` schema and know nothing about the source. To swap in a live feed later, only `load_reddit_archive.py` changes.

## 1. Setup

In [ ]:
from __future__ import annotations

import logging
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Repository root is the parent of `notebooks/`.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.load_reddit_archive import ArchiveConnector
from src.link_comments_to_matches import LinkingConfig, LinkerOptions, MatchLinker
from src.sentiment import VaderAnalyzer, score_dataframe
from src.engagement import (
    EngagementWeights,
    aggregate_per_match,
    compute_engagement_score,
    join_with_stage2,
)

# Quiet notebook output.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Paths.
RAW_REDDIT_DIR        = REPO_ROOT / "data" / "raw" / "reddit"
INTERIM_COMMENTS_PQ   = REPO_ROOT / "data" / "interim" / "reddit_comments.parquet"
STAGE2_MATCHES_CSV    = REPO_ROOT / "outputs" / "stage2_matches.csv"
STAGE2_ENGAGEMENT_CSV = REPO_ROOT / "outputs" / "stage2_engagement.csv"
CONFIG_YAML           = REPO_ROOT / "config" / "team_aliases.yaml"
PROCESSED_DIR         = REPO_ROOT / "outputs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Pushshift dumps dir:", RAW_REDDIT_DIR, "(exists:", RAW_REDDIT_DIR.exists(), ")")
print("Interim comments parquet:", INTERIM_COMMENTS_PQ, "(exists:", INTERIM_COMMENTS_PQ.exists(), ")")

## 2. Load Stage 2 outputs

Stage 2 is upstream and authoritative — we don't modify it, we consume it. The minimum contract we require:

- `stage2_matches.csv` has `match_id`, `home_team`, `away_team`, `match_date`.
- `stage2_engagement.csv` has `match_id` and one engagement-score column.

If either file is missing the notebook bails early with a clear pointer — far better than silently failing later.

In [ ]:
for required in (STAGE2_MATCHES_CSV, STAGE2_ENGAGEMENT_CSV):
    if not required.exists():
        raise FileNotFoundError(
            f"Missing Stage 2 output: {required}\n"
            f"Place stage2_*.csv files in data/processed/ before running Stage 3."
        )

matches = pd.read_csv(STAGE2_MATCHES_CSV)
stage2_engagement = pd.read_csv(STAGE2_ENGAGEMENT_CSV)

print(f"Stage 2 matches:    {len(matches):>6} rows | columns: {list(matches.columns)}")
print(f"Stage 2 engagement: {len(stage2_engagement):>6} rows | columns: {list(stage2_engagement.columns)}")
matches.head(3)

## 3. Load Reddit comments (materialized from Pushshift archives)

The expensive bit — parsing the zstandard NDJSON dumps — was done offline by `python -m src.load_reddit_archive`. The notebook just loads the pre-materialized parquet. This keeps notebook runtime fast and reproducible.

If the parquet is missing, the cell falls back to materializing it now from whatever .zst files are in `data/raw/reddit/` — useful for first-time runs but slower.

In [ ]:
if INTERIM_COMMENTS_PQ.exists():
    comments = pd.read_parquet(INTERIM_COMMENTS_PQ)
    print(f"Loaded {len(comments):,} comments from cached parquet.")
else:
    print("No cached parquet — materializing from .zst archives. This may take a while.")
    if not any(RAW_REDDIT_DIR.glob("*.zst")):
        raise FileNotFoundError(
            f"No .zst files in {RAW_REDDIT_DIR}. "
            f"See README_stage3.md for dataset placement instructions."
        )
    connector = ArchiveConnector(
        input_dir=RAW_REDDIT_DIR,
        min_body_length=15,
    )
    connector.materialize(INTERIM_COMMENTS_PQ)
    comments = pd.read_parquet(INTERIM_COMMENTS_PQ)
    print(f"Materialized and loaded {len(comments):,} comments.")

comments.head(3)

## 4. Link comments to matches

Keyword + temporal window join. Tuneable knobs:

- `window_hours_before` / `window_hours_after` — pre-match build-up and post-match reaction window.
- `min_confidence` — filter out weak links. Defaults to 0.4 (roughly: "team mentioned + sub is a generic football sub" = 0.45 floor).

Output is many-to-many — a comment can link to multiple matches if it mentions a team that played twice in the window. We let downstream aggregation handle that.

In [ ]:
cfg = LinkingConfig.from_yaml(CONFIG_YAML)
linker = MatchLinker(
    cfg,
    LinkerOptions(window_hours_before=48, window_hours_after=48, min_confidence=0.4),
)
links = linker.link(comments, matches)

print(f"\nLink table: {len(links):,} (match, comment) pairs across {links['match_id'].nunique():,} matches")
print("\nConfidence distribution:")
print(links["link_confidence"].describe().round(3))
print("\nTop reasons (rough count of co-occurring rules):")
(
    links["link_reasons"].str.split(";").explode().value_counts().head(10)
)

## 5. Sentiment scoring

VADER baseline. The `SentimentAnalyzer` interface in `src/sentiment.py` accepts any analyzer that returns a `[-1, 1]` float, so a transformer can be dropped in later without touching this notebook.

We score the *full* comments table, not just the linked subset — this way, if we later widen the time window or add teams, we don't have to re-score.

In [ ]:
analyzer = VaderAnalyzer()
scored_comments = score_dataframe(comments, text_col="body", analyzer=analyzer)

print("Sentiment distribution (VADER compound):")
print(scored_comments["sentiment"].describe().round(3))
scored_comments[["id", "subreddit", "sentiment"]].head(5)

## 6. Aggregate to per-match engagement

Features per match:

| Feature | Interpretation |
| --- | --- |
| `comment_count` | Raw discussion volume |
| `weighted_comment_count` | Volume discounted by link confidence |
| `avg_sentiment` | Net mood (sign matters) |
| `sentiment_volatility` | Std — **high = fanbases disagree = excitement signal** |
| `positive_ratio`, `negative_ratio` | Asymmetric mood split |
| `peak_hour_count` | Peakedness around kickoff |
| `unique_subreddits` | Cross-community reach |

`engagement_score` is a transparent linear combination of min-max-scaled components — easy to audit, easy to argue with.

In [ ]:
per_match = aggregate_per_match(links, scored_comments)
per_match = compute_engagement_score(
    per_match,
    weights=EngagementWeights(volume=0.45, affect=0.20, volatility=0.25, reach=0.10),
)

print(f"Per-match aggregate: {len(per_match):,} matches")
per_match.sort_values("engagement_score", ascending=False).head(10)

## 7. Join with Stage 2 football-side engagement

Now we have two independent signals: **football** (Stage 2 — goals, xG, tempo) and **fans** (Stage 3 — Reddit sentiment, volatility). They measure different things and *should* sometimes disagree — a 0-0 with two world-class keepers is high-engagement on the fan side, low on the football side. Tracking both is the point.

In [ ]:
combined = join_with_stage2(per_match, stage2_engagement)
print(f"Combined frame: {len(combined):,} matches")
combined.head(5)

## 8. Where do the signals agree vs disagree?

Useful sanity check. If the two signals are uncorrelated, the combined score is meaningful; if they're identical, fan-side adds nothing. We expect a moderate positive correlation.

In [ ]:
valid = combined.dropna(subset=["engagement_score_football_raw", "engagement_score_fans"])
valid = valid[valid["fan_comment_count"] > 0]  # only matches with fan signal

if len(valid) > 1:
    corr = valid["engagement_score_football_raw"].corr(valid["engagement_score_fans"])
    print(f"Correlation (n={len(valid)}): {corr:+.3f}")
else:
    print("Not enough overlap to compute correlation.")

# Matches where fans were excited but the football was dull, and vice-versa.
if len(valid) >= 5:
    valid = valid.assign(divergence=valid["engagement_score_fans"] - valid["engagement_score_football_raw"])
    print()
    print("Top 5 'fans more excited than the football' matches:")
    cols = ["match_id", "engagement_score_football_raw", "engagement_score_fans", "divergence", "fan_comment_count"]
    print(valid.nlargest(5, "divergence")[cols].to_string(index=False))
    print()
    print("Top 5 'football wild but fans quiet' matches:")
    print(valid.nsmallest(5, "divergence")[cols].to_string(index=False))

## 9. Save Stage 3 outputs

In [ ]:
(PROCESSED_DIR / "stage3_comments_linked.parquet").parent.mkdir(parents=True, exist_ok=True)

links.to_parquet(PROCESSED_DIR / "stage3_comments_linked.parquet", index=False)
per_match.to_csv(PROCESSED_DIR / "stage3_match_sentiment.csv", index=False)
combined.to_csv(PROCESSED_DIR / "stage3_engagement_enriched.csv", index=False)

# Stage 3 ranking — sorted by combined score with full match metadata.
ranking_base = combined.sort_values("engagement_score_combined", ascending=False).reset_index(drop=True)
ranking_base.insert(0, "rank", ranking_base.index + 1)

# Merge team names, dates, and scores from Stage 2 match metadata.
meta_cols = ["match_id", "match_date", "home_team", "away_team", "home_score", "away_score", "is_rivalry", "total_goals"]
ranking = ranking_base.merge(matches[meta_cols], on="match_id", how="left")
ranking["score_str"] = ranking["home_score"].astype(int).astype(str) + "-" + ranking["away_score"].astype(int).astype(str)

ranking_cols = [
    "rank", "match_id", "match_date", "home_team", "away_team", "score_str",
    "total_goals", "is_rivalry", "fan_comment_count",
    "engagement_score_fans", "engagement_score_football_raw",
    "engagement_score_football_norm", "engagement_score_combined",
]
ranking = ranking[[c for c in ranking_cols if c in ranking.columns]]
ranking.to_csv(PROCESSED_DIR / "stage3_ranking.csv", index=False)

print("Written to outputs/:")
for p in [
    "stage3_comments_linked.parquet",
    "stage3_match_sentiment.csv",
    "stage3_engagement_enriched.csv",
    "stage3_ranking.csv",
]:
    fp = PROCESSED_DIR / p
    print(f"  {p:<40} {fp.stat().st_size / 1024:>8.1f} KB")

## 10. Top excited matches by combined score

In [ ]:
display_cols = [
    "rank", "match_id",
    "engagement_score_combined", "engagement_score_football_raw", "engagement_score_football_norm",
    "engagement_score_fans", "fan_comment_count",
]
ranking[[c for c in display_cols if c in ranking.columns]].head(15)

---

## Assumptions, caveats, and next steps

**Assumptions baked into this stage:**

1. *Comment is about the match in question* — keyword + temporal window is a proxy, not a guarantee. Words like "Barca" can refer to the basketball team or the Catalan football club. The `link_confidence` column is the honest knob — raise the threshold to trade recall for precision.

2. *VADER sentiment generalizes to football discussion* — VADER was tuned on social-media-style text, but it does not know football idioms ("binned", "shocker", "bottling"). A domain-adapted classifier would do better; the `SentimentAnalyzer` interface is in place for that upgrade.

3. *Reddit fans are a representative slice of fan sentiment* — they aren't. Reddit skews Anglophone, young, and toward certain fanbases (r/Gunners has 1M+ subs, smaller clubs have a few thousand). Anywhere we compare across clubs we should normalize for subreddit size, not just count comments.

**Natural next steps:**

- Replace `VaderAnalyzer` with `cardiffnlp/twitter-roberta-base-sentiment-latest` and rerun.
- Add a submissions ingestion path (`RS_*.zst` dumps) — submission titles are often higher-signal than comments.
- Train a small (comment, match) link classifier on hand-labeled pairs to replace the rule-based confidence scorer.
- Add a `LiveConnector` implementing `BaseRedditConnector` for whatever Reddit API surface is stable at that time.